In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch
import json
import gc
import outlines #https://dottxt-ai.github.io/outlines/latest/
from pydantic import BaseModel
from typing import List

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

print (device)

cuda


<hr>
<hr>

In [ ]:
quant = BitsAndBytesConfig (
    load_in_4bit = True,  #Quantização
    bnb_4bit_quant_type = 'nf4',  #Tipo de Quantização
    bnb_4bit_use_double_quant = True, #Double Quantization
    bnb_4bit_compute_dtype = 'float16', #Tipo de precisão usada nos cálculos durante a inferência.
)


modelo = AutoModelForCausalLM.from_pretrained (r"microsoft/phi-4", quantization_config = quant, device_map = device)
tokenizer = AutoTokenizer.from_pretrained (r"microsoft/phi-4")

modelo.save_pretrained (r"C:\Users\Admin\Desktop\models\microsoftphi-4-Q4")
tokenizer.save_pretrained (r"C:\Users\Admin\Desktop\models\microsoftphi-4-Q4")

<hr>
<hr>

In [3]:
#path = r"C:\Users\Admin\Desktop\models\microsoftphi-4-Q4"

path = r"C:\Users\Admin\Desktop\models\Mistral-7B-Instruct-v0.3-Q4"

model = AutoModelForCausalLM.from_pretrained (path, device_map = device)
tokenizer = AutoTokenizer.from_pretrained (path)

Loading weights: 100%|██████████| 291/291 [00:03<00:00, 79.17it/s] 


In [ ]:
prompt = "Olá, quem foi o primeiro rei de Portugal ?"

tokens = tokenizer (prompt, return_tensors = "pt").to (device)

with torch.no_grad():
    logits = model.generate (**tokens, max_new_tokens = 512)

input_length = tokens["input_ids"].shape[1]
response_tokens = logits[0][input_length:]

output = tokenizer.decode(response_tokens, skip_special_tokens = True)

<hr>
<hr>

In [4]:
from unstructured.partition.pdf import partition_pdf
from unstructured.chunking.title import chunk_by_title

from pathlib import Path
from langchain_core.documents import Document


In [5]:
path = Path (r"C:\Users\Admin\Desktop\ip\How To RAG\docs")

documentos = []
chunk_id = 1

for docs in path.iterdir():

    file_dtype = docs.suffix.lower()

    if file_dtype == ".pdf":

        parsing = partition_pdf (str(docs), languages = ["por"])
        chunks = chunk_by_title (parsing[3:], max_characters = 750)

    '''elif file_dtype == ".txt":

        parsing = partition_text (str(docs), languages = ["por"])
        chunks = chunk_elements (parsing, max_characters = 750)

    elif file_dtype == ".docx":        
        
        parsing = partition_docx (str(docs), languages = ["por"])
        chunks = chunk_by_title (parsing, max_characters = 750)'''


    for chunk in chunks:

        documentos.append (
            Document (
                page_content = chunk.text,
                metadata = {
                    "title": docs.name,
                    "chunk_id": chunk_id
                }
            )
        )

        chunk_id += 1


#documentos

<hr>
<hr>

In [ ]:
x = 0
y = 15

model.eval()
for i in range (len(documentos) // 15):


    prompt = f"""
    <|im_start|>system<|im_sep|>

    És um assistente de IA com a função de Senior Data Scientist que tens como único e principal objetivo 
    criar um dataset sintético de acordo com os dados fornecidos. Deves criar perguntas de acordo com o 
    contexto e selecionar quais são os chunks mais relevantes para responder a essas perguntas.

    <REGRAS>

    1. És um modelo que retorna outputs apenas em formato JSON. Qualquer coisa retornada num formato diferente está errada!
    2. Deves ter em conta e ser fiel aos dados fornecidos.
    3. Cria perguntas concisas e diretas de acordo com o contexto fornecido.
    4. Deves ter em conta o 'chunk_id' de cada contexto e selecionar quais os 'chunk_id' mais relevantes para responder à pergunta criada. Tenta sempre selecionar mais que um 'chunk_id'
    5. Por cada contexto fornecido retorna no máximo 5 pares de perguntas e 'chunk_id' ideal para responder à pergunta.

    </REGRAS>


    <EXEMPLO_CONTEXTO>

    'chunk_id': 1, 'title': 'ABC_Máquinas_Elétricas.pdf', 'content': 'Actualmente, podemos considerar as máquinas eléctricas (motores, geradores e transformadores)'

    </EXEMPLO_CONTEXTO>


    <EXEMPLO_OUTPUT>

        "query": "Que tipos de máquinas elétricas existem ?",
        "chunk_id": [1, 2]

    </EXEMPLO_OUTPUT> 

    Tens como principal e única função responder de acordo com o contexto fornecido e criar perguntas 
    de acordo com o contexto e selecionar os chunk_id mais relevantes para responder à pergunta
    criada. Lembra-te que é fulcral que o output seja em formato JSON.
    <|im_end|>


    <|im_start|>user<|im_sep|>

    {documentos[x:y]}

    <|im_end|>


    <|im_start|>assistant<|im_sep|>


    """

    tokens = tokenizer (prompt, return_tensors = "pt").to (device)

    with torch.no_grad():
        logits = model.generate (**tokens, max_new_tokens = 180, use_cache = True)

    input_length = tokens["input_ids"].shape[1]
    response_tokens = logits[0][input_length:]

    output = tokenizer.decode(response_tokens, skip_special_tokens = True)

    with open ("dataset.json", "a", encoding = "utf-8") as f:
        f.write (output + "\n")

    x = y
    y = y + 15

    del tokens, logits, output
    #gc.collect()
    torch.cuda.empty_cache()

    print (f"{i} Done")

    '''prompt_2 = f"""
    <|im_start|>system<|im_sep|>

    És um assistente de IA com a única e exclusiva função de avaliar o output que um modelo deu ao criar Dados Sintéticos.
    Vais receber o contexto e vais funcionar como um Avaliador do output gerado.

    <REGRAS>

    1. É fulcral e obrigatório que retornes o output no formato pretendido.
    2. Deves confirmar sempre se o chunk_id bate certo com a pergunta criada. É fulcral porque caso a resposta não esteja no chunk_id selecionado, o dataset fica enviesado.
    3. Vais avaliar as seguintes métricas:
        Fidelidade ao Contexto: A resposta à pergunta está contida no chunk ?
        Diversidade Semântica: A pergunta cobre vários chunks ?
    4. Todas as métricas devem ser avaliadas de maneira binária "Sim" ou "Não. Tudo o que seja contrário é isto, é considerado erro!

    </REGRAS>


    <EXEMPLO_CONTEXTO>

    'chunk_id': 1, 'title': 'ABC_Máquinas_Elétricas.pdf', 'content': 'Actualmente, podemos considerar as máquinas eléctricas (motores, geradores e transformadores)'
    "query": "Em que locais podemos encontrar geradores?", "chunk_id": [2]

    </EXEMPLO_CONTEXTO>


    <EXEMPLO_OUTPUT>

        "Fidelidade ao Contexto": "Sim"
        "Diversidade Semântica": "Não"

    </EXEMPLO_OUTPUT> 

    Tens como principal e única função responder de acordo com o contexto fornecido e fornecer uma avaliação fiel e fidedigna.
    Lembra-te que deves retornar a resposta no formato exigido.

    <|im_end|>


    <|im_start|>user<|im_sep|>

    {documentos[x:y]}
    {output}

    <|im_end|>


    <|im_start|>assistant<|im_sep|>

    """

    x = y
    y = y + 15

    tokens = tokenizer (prompt_2, return_tensors = "pt").to (device)

    with torch.no_grad():
        logits = model.generate (**tokens, max_new_tokens = 256, use_cache = True)

    input_length = tokens["input_ids"].shape[1]
    response_tokens = logits[0][input_length:]

    output = tokenizer.decode(response_tokens, skip_special_tokens = True)

    print (output)
    print ("---" *50)
    print (x, y)
'''


In [6]:
class PARES (BaseModel):
    query: str
    chunk_id: List[int]

class DATASET (BaseModel):
    data: List[PARES]


model = outlines.from_transformers (model, tokenizer)


<h1> Este prompt funcionou! Perceber se é do prompt ou das class! Construir Dataset </h1>

<h2> LLM EVAL </h2>

In [7]:
x = 0
y = 15


for i in range (1):

    prompt = f"""
    <s>system

    És um assistente de IA com a função de Senior Data Scientist que tens como único e principal objetivo 
    criar um dataset sintético de acordo com os dados fornecidos. Deves criar perguntas de acordo com o 
    contexto e selecionar quais são os chunks mais relevantes para responder a essas perguntas.

    <REGRAS>

    1. És um modelo que retorna outputs apenas em formato JSON. Qualquer coisa retornada num formato diferente está errada!
    2. Deves ter em conta e ser fiel aos dados fornecidos.
    3. Cria perguntas concisas e diretas de acordo com o contexto fornecido.
    4. Deves ter em conta o 'chunk_id' de cada contexto e selecionar quais os 'chunk_id' mais relevantes para responder à pergunta criada. Tens que selecionar sempre mais que um 'chunk_id'!!
    5. Não podes repetir perguntas e retorna apenas 5 exemplos por prompt!
    6. Segue sempre este formato!
        query: str
        chunk_id: List[int] AQUI TEM QUE HAVER MAIS QUE UM CHUNK_ID!!

    </REGRAS>


    <INPUT>

    'chunk_id': 1, 'title': 'ABC_Máquinas_Elétricas.pdf', 'content': 'Actualmente, podemos considerar as máquinas eléctricas (motores, geradores e transformadores)'

    </INPUT>


    <OUTPUT>

        "query": "Que tipos de máquinas elétricas existem ?",
        "chunk_id": [1, 2]

        "query": "O que é eletricidade ?",
        "chunk_id": [3, 8, 9]

        "query": "Que tipos de corrente existem ?",
        "chunk_id": [10, 20, 230]

    </OUTPUT

    
    Tens como principal e única função responder de acordo com o contexto fornecido e criar perguntas 
    de acordo com o contexto e selecionar os chunk_id mais relevantes para responder à pergunta
    criada. Lembra-te que é fulcral que o output seja em formato JSON.
    </s>

    
    <s>user

    <CONTEXTO>

    {documentos[x:y]}

    </CONTEXTO
    </s>

    
    <s>assistant

    """

    result = model (prompt, output_type = DATASET, max_new_tokens = 500)

    with open ("dataset-Mistral.json", "a", encoding = "utf-8") as f:
        f.write (result + "\n") 
    
    x = y
    y = y + 15

    torch.cuda.empty_cache()

[transformers] Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
W0703 17:26:27.280000 17736 Lib\site-packages\torch\_dynamo\convert_frame.py:1125] WON'T CONVERT _apply_token_bitmask_inplace_kernel c:\Users\Admin\Desktop\ip\How To RAG\.venv\Lib\site-packages\outlines_core\kernels\torch.py line 43 
W0703 17:26:27.280000 17736 Lib\site-packages\torch\_dynamo\convert_frame.py:1125] due to: 
W0703 17:26:27.280000 17736 Lib\site-packages\torch\_dynamo\convert_frame.py:1125] Traceback (most recent call last):
W0703 17:26:27.280000 17736 Lib\site-packages\torch\_dynamo\convert_frame.py:1125]   File "c:\Users\Admin\Desktop\ip\How To RAG\.venv\Lib\site-packages\torch\_dynamo\output_graph.py", line 1446, in _call_user_compiler
W0703 17:26:27.280000 17736 Lib\site-packages\torch\_dynamo\convert_frame.py:1125]     compiled_fn = compiler_fn(gm, self.example_inputs())
W0703 17:26:27.280000 17736 Lib\site-packages\torch\_dynamo\convert_frame.py:1125]                   ^^^^^^^^^^^^^^